# 02 · Linear regression and gradient descent

**Goal:** solve the same problem two ways (closed form vs. iterative), and see *why* gradient descent's speed depends on the eigenvalues of the Hessian.

Model $\hat y = Xw$ with loss

$$L(w) = \frac{1}{2n}\|Xw-y\|^2,\qquad \nabla L(w)=\frac{1}{n}X^\top(Xw-y),\qquad \nabla^2 L = \frac{1}{n}X^\top X =: H.$$

Setting the gradient to zero gives the **normal equations** $X^\top X w = X^\top y$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
n, d = 200, 3
X = np.c_[np.ones(n), rng.normal(size=(n, d - 1))]     # first column = bias
w_true = np.array([1.0, 2.0, -3.0])
y = X @ w_true + 0.3 * rng.normal(size=n)

def loss(w, X=X, y=y):
    r = X @ w - y
    return 0.5 * np.mean(r**2)

def grad(w, X=X, y=y):
    return X.T @ (X @ w - y) / len(y)

w_ne = np.linalg.solve(X.T @ X, X.T @ y)               # use solve, not inv
print("normal equations:", w_ne.round(4))
assert np.allclose(grad(w_ne), 0, atol=1e-10)

## Gradient descent and the step size

$w_{k+1}=w_k-\eta\nabla L(w_k)$. For a quadratic the error evolves as $e_{k+1}=(I-\eta H)e_k$, so in the eigenbasis of $H$ each component shrinks by $|1-\eta\lambda_i|$ per step.

* Converges iff $0<\eta<2/\lambda_{\max}$.
* With $\eta=1/\lambda_{\max}$ the slowest component shrinks by $1-1/\kappa$ where $\kappa=\lambda_{\max}/\lambda_{\min}$ is the **condition number**.

In [ ]:
H = X.T @ X / n
lam = np.linalg.eigvalsh(H)
Lmax, mu = lam.max(), lam.min()
print("eigenvalues of H:", lam.round(4), "  kappa =", round(Lmax / mu, 3))

def gd(eta, steps, w0=None):
    w = np.zeros(d) if w0 is None else w0.copy()
    hist = []
    for _ in range(steps):
        hist.append(loss(w))
        w = w - eta * grad(w)
    return w, np.array(hist)

w1, h1 = gd(1.0 / Lmax, 200)
assert np.allclose(w1, w_ne, atol=1e-6)

# too-large step: eta > 2 / Lmax diverges
_, h_bad = gd(2.1 / Lmax, 60)
print("loss with eta = 2.1/Lmax: start", h_bad[0].round(3), " end", h_bad[-1].round(1))
assert h_bad[-1] > h_bad[0]

plt.semilogy(h1 - loss(w_ne) + 1e-16)
plt.xlabel("iteration"); plt.ylabel("L(w_k) - L*"); plt.title("GD, eta = 1/Lmax")
plt.show()

## Bad conditioning slows GD

Rescale one feature by 10. The Hessian eigenvalues spread out (about $100\times$), so GD with the best safe step needs roughly $\kappa$ times more iterations. This is why **feature scaling** matters in practice.

In [ ]:
def iters_to_tol(Xs, tol=1e-6, cap=50000):
    ns = len(y)
    Hs = Xs.T @ Xs / ns
    ev = np.linalg.eigvalsh(Hs)
    eta = 1.0 / ev.max()
    w_star = np.linalg.solve(Xs.T @ Xs, Xs.T @ y)
    w = np.zeros(Xs.shape[1])
    for k in range(cap):
        if np.linalg.norm(w - w_star) < tol:
            return k, ev.max() / ev.min()
        w = w - eta * Xs.T @ (Xs @ w - y) / ns
    return cap, ev.max() / ev.min()

X_bad = X.copy()
X_bad[:, 2] *= 10.0

k_good, kap_good = iters_to_tol(X)
k_bad, kap_bad = iters_to_tol(X_bad)
print(f"well scaled : kappa = {kap_good:8.1f}  iterations = {k_good}")
print(f"bad scaling : kappa = {kap_bad:8.1f}  iterations = {k_bad}")
assert k_bad > 5 * k_good

## Exercises

1. Add a standardization step (subtract mean, divide by std) and confirm $\kappa$ drops.
2. Implement **ridge regression** $L(w)+\tfrac{\lambda}{2}\|w\|^2$. Derive its closed form $(X^\top X+n\lambda I)w=X^\top y$ and check $\nabla L=0$ there.
3. Implement mini-batch SGD with batch size 16 and a decaying step size. Plot its loss next to full-batch GD.